In [7]:
import sys
from pathlib import Path
import warnings

import torch
import torch.nn.functional as F
from sklearn.exceptions import ConvergenceWarning

# ---------------- ROOT + sys.path ----------------
ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.codi.models import CODI

warnings.filterwarnings("ignore", category=ConvergenceWarning)          # sklearn MLP
warnings.filterwarnings("ignore", message="Parameters: {")              # XGBoost unused params
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# ---------------- Patch GaussianDiffusionTrainer + Sampler ----------------
from katabatic.models.codi.utils import (
    GaussianDiffusionTrainer,
    GaussianDiffusionSampler,
    extract,
)

def _make_x_t(self, x_0, t, noise):
    # same formula as trainer forward()
    x_t = (
        extract(self.sqrt_alphas_bar, t, x_0.shape) * x_0 +
        extract(self.sqrt_one_minus_alphas_bar, t, x_0.shape) * noise
    )
    return x_t

GaussianDiffusionTrainer.make_x_t = _make_x_t

def _p_mean_variance(self, x_t, t, cond):
    """
    Compute mean and log-variance for p(x_{t-1} | x_t).
    CoDi calls this in _sample_reverse.
    """
    # predict epsilon with the model
    eps = self.model(x_t, t, cond)

    # reconstruct x_0 prediction
    x_0_pred = (
        extract(self.sqrt_recip_alphas_bar, t, x_t.shape) * x_t -
        extract(self.sqrt_recipm1_alphas_bar, t, x_t.shape) * eps
    )
    x_0_pred = torch.clamp(x_0_pred, -1, 1)

    # posterior mean and log variance (already precomputed in buffers)
    mean = (
        extract(self.posterior_mean_coef1, t, x_t.shape) * x_0_pred +
        extract(self.posterior_mean_coef2, t, x_t.shape) * x_t
    )
    log_variance = extract(self.posterior_log_variance, t, x_t.shape)

    return mean, log_variance

GaussianDiffusionSampler.p_mean_variance = _p_mean_variance
print("✅ Patched GaussianDiffusionTrainer.make_x_t and GaussianDiffusionSampler.p_mean_variance")

# ---------------- Patch CODI._train_step (one-hot cond) ----------------
from katabatic.models.codi.models import CODI as _CODI

def _codi_train_step_patched(
    self,
    x_con: torch.Tensor,
    x_cat: torch.Tensor,
    optim_con: torch.optim.Optimizer,
    optim_dis: torch.optim.Optimizer,
):
    """Single training step with corrected conditioning (one-hot for continuous branch)."""
    batch_size = x_con.shape[0]
    device = self.device

    # Sample timesteps
    t = torch.randint(0, self.n_steps, (batch_size,), device=device)

    loss_con_total = torch.tensor(0.0, device=device)
    loss_dis_total = torch.tensor(0.0, device=device)

    # ===== Continuous diffusion loss (if we have continuous features) =====
    if self.has_continuous_:
        noise_con = torch.randn_like(x_con)
        x_t_con = self.trainer_con_.make_x_t(x_con, t, noise_con)

        # ✅ use one-hot categorical as condition (dim = sum(num_classes_))
        cond_cat = self._to_onehot(x_cat)

        # Diffusion loss
        eps_pred = self.model_con_(x_t_con, t, cond_cat)
        loss_con_diff = F.mse_loss(eps_pred, noise_con)

        # Contrastive part (also using one-hot)
        neg_indices = torch.randperm(batch_size, device=device)
        x_cat_neg = x_cat[neg_indices]
        cond_cat_neg = self._to_onehot(x_cat_neg)

        eps_pos = self.model_con_(x_t_con, t, cond_cat)
        eps_neg = self.model_con_(x_t_con, t, cond_cat_neg)

        loss_con_contrast = torch.relu(
            F.mse_loss(eps_neg, noise_con, reduction="none").mean()
            - F.mse_loss(eps_pos, noise_con, reduction="none").mean()
            + 1.0
        )

        loss_con_total = loss_con_diff + self.lambda_con * loss_con_contrast

        optim_con.zero_grad()
        loss_con_total.backward()
        torch.nn.utils.clip_grad_norm_(self.model_con_.parameters(), self.grad_clip)
        optim_con.step()

    # ===== Discrete diffusion loss (if we have categorical features) =====
    if self.has_categorical_:
        x_cat_onehot = self._to_onehot(x_cat)
        log_x_start = torch.log(x_cat_onehot.float().clamp(min=1e-30))
        x_t_cat = self.trainer_dis_.q_sample(log_x_start, t)
        kl, _ = self.trainer_dis_.compute_Lt(log_x_start, x_t_cat, t, x_con)
        kl_prior = self.trainer_dis_.kl_prior(log_x_start)
        loss_dis_total = (kl + kl_prior).mean()

        optim_dis.zero_grad()
        loss_dis_total.backward()
        torch.nn.utils.clip_grad_norm_(self.model_dis_.parameters(), self.grad_clip)
        optim_dis.step()

    return loss_con_total.item(), loss_dis_total.item()

_CODI._train_step = _codi_train_step_patched
print("✅ Patched CODI._train_step to use one-hot conditioning")

# ---------------- Dataset + paths (ADULT) ----------------
dataset_name = "adult"

dataset_path = ROOT / "raw_data" / f"{dataset_name}.csv"
output_path = ROOT / "discretized_data" / f"{dataset_name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Discretizing {dataset_path} -> {output_path}")
discretize_preprocess(str(dataset_path), str(output_path))

input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "codi")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

# ---------------- CoDi model for Adult ----------------
class CoDiAdult(CODI):
    def __init__(self):
        super().__init__(
            n_steps=50,
            beta_1=1e-5,
            beta_T=0.02,
            encoder_dim_con=(64, 128, 256),
            encoder_dim_dis=(64, 128, 256),
            nf_con=16,
            nf_dis=64,
            activation="relu",
            epochs=30,
            batch_size=512,
            lr_con=2e-3,
            lr_dis=2e-3,
            grad_clip=1.0,
            lambda_con=0.2,
            lambda_dis=0.2,
            random_state=42,
            device=None,   # auto: cuda if available
        )

pipeline = TrainTestSplitPipeline(
    model=lambda: CoDiAdult()
)

# 🔧 PATCH SKLEARN NAME CHECK HERE
import sklearn.utils.validation as skval

def _no_check_feature_names(estimator, X, reset=True):
    # Ignore sklearn's feature name check (we handle alignment ourselves)
    return

skval._check_feature_names = _no_check_feature_names
print("✅ Patched sklearn._check_feature_names to ignore column name order")

# 🔧 PATCH XGBoost FEATURE CHECK HERE
import xgboost.core as xgcore

def _no_validate_features(self, feature_names):
    # Ignore XGBoost's feature name validation (order-only differences)
    return

xgcore.Booster._validate_features = _no_validate_features
print("✅ Patched xgboost.Booster._validate_features to ignore feature name order")

# ---------------- Run pipeline ----------------
result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print("\nPipeline result (should include TSTR metrics + result path):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
✅ Patched GaussianDiffusionTrainer.make_x_t and GaussianDiffusionSampler.p_mean_variance
✅ Patched CODI._train_step to use one-hot conditioning
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\adult.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\adult.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\adult.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\adult.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\adult.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\adult
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\adult
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\adult\codi
✅ Patched sklearn._check_feature_names to ignore column name order
✅ Patched xgboost.Booster._validate_features to ignore feature name order
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (2604

INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Training CoDi Model
INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Loaded training data: (26048, 15)
INFO:katabatic.models.codi.models:Schema: 1 continuous, 14 categorical columns


Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)


INFO:katabatic.models.codi.models:Building models: con_dim=1, cat_dim=117
INFO:katabatic.models.codi.models:Continuous model params: 166,519
INFO:katabatic.models.codi.models:Discrete model params: 372,905
INFO:katabatic.models.codi.models:
Training for 30 epochs...
INFO:katabatic.models.codi.models:Epoch 1/30: loss_con=0.7640, loss_dis=121.9899
INFO:katabatic.models.codi.models:Epoch 5/30: loss_con=0.4275, loss_dis=119.3686
INFO:katabatic.models.codi.models:Epoch 10/30: loss_con=0.4096, loss_dis=118.8677
INFO:katabatic.models.codi.models:Epoch 15/30: loss_con=0.4017, loss_dis=118.7051
INFO:katabatic.models.codi.models:Epoch 20/30: loss_con=0.3829, loss_dis=118.5697
INFO:katabatic.models.codi.models:Epoch 25/30: loss_con=0.3913, loss_dis=118.4782
INFO:katabatic.models.codi.models:Epoch 30/30: loss_con=0.3865, loss_dis=118.4117
INFO:katabatic.models.codi.models:
Generating 26048 synthetic samples...
INFO:katabatic.models.codi.models:
Synthetic data saved to: C:\Users\Prabu\Downloads\Kat


Results saved to: Results\adult\codi_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7645
F1 Score: 0.6675
AUC: 0.4163

MLP:
Accuracy: 0.7089
F1 Score: 0.6400
AUC: 0.5089

RF:
Accuracy: 0.7593
F1 Score: 0.6553
AUC: 0.4458

XGBoost:
Accuracy: 0.7593
F1 Score: 0.6553
AUC: 0.5378

Pipeline result (should include TSTR metrics + result path):
Train test split pipeline executed successfully.
